In [8]:
%%time
%matplotlib inline
import rioxarray
import numpy as np
import importlib 
import matplotlib.pyplot as plt
import geopandas as gpd

CPU times: total: 0 ns
Wall time: 16.5 ms


In [10]:
ndvi_data = rioxarray.open_rasterio('../input/STNDVI_202223.tif')
vh_data = rioxarray.open_rasterio('../input/STVH_202223.tif')
import rioxarray

input_path = "../input/ST_lua-tom.tif"
mask_data = rioxarray.open_rasterio(input_path)
mask_data

<xarray.DataArray (band: 1, y: 3323, x: 4502)> Size: 15MB
[14960146 values with dtype=uint8]
Coordinates:
  * band         (band) int64 8B 1
  * x            (x) float64 36kB 5.83e+05 5.83e+05 ... 6.28e+05 6.28e+05
  * y            (y) float64 27kB 1.06e+06 1.06e+06 ... 1.027e+06 1.027e+06
    spatial_ref  int64 8B 0
Attributes:
    AREA_OR_POINT:       Area
    RepresentationType:  THEMATIC
    _FillValue:          255
    scale_factor:        1.0
    add_offset:          0.0

In [11]:
import xarray as xr
import numpy as np

# Assuming 'data' is your xarray.DataArray
mask = mask_data != 255  # Boolean mask where values are not 255

# Stack x and y dimensions into a single index
data_stacked = mask_data.stack(points=("y", "x"))

# Apply the mask and drop NaN values
filtered = data_stacked.where(data_stacked != 255, drop=True)

# Extract x and y coordinates
x_indices = filtered.coords["x"].values
y_indices = filtered.coords["y"].values

# Stack into a (N, 2) NumPy array
xy_valid = np.column_stack((x_indices, y_indices))

print(xy_valid)  # Print or use the filtered coordinates

[[ 611462.754  1059985.3978]
 [ 611472.754  1059985.3978]
 [ 611482.754  1059985.3978]
 ...
 [ 591172.754  1026775.3978]
 [ 591162.754  1026765.3978]
 [ 591172.754  1026765.3978]]


In [12]:
i = 0
ndvi_list = []
vh_list = []
for index in xy_valid:
    data = ndvi_data.sel(x=index[0], y=index[1], method='nearest').values
    vh = vh_data.sel(x=index[0], y=index[1], method='nearest').values
    if not np.any(np.isnan(data)):
        ndvi_list.append(data)
        vh_list.append(vh)

In [5]:
len(ndvi_list)

0

In [6]:

# Lọc các mảng không chứa nan
filtered_arrays = [arr for arr in ndvi_list if not np.any(np.isnan(arr))]

# In kết quả
if filtered_arrays:
    for arr in filtered_arrays:
        print(arr)
else:
    print("Không có mảng nào trong dữ liệu thỏa mãn điều kiện (không chứa nan).")

Không có mảng nào trong dữ liệu thỏa mãn điều kiện (không chứa nan).


In [7]:

np.save('../input/nonan_lt_ndvi.npy',ndvi_list)
np.save('../input/nonan_lt_vh.npy',vh_list)
len(vh_list)


0